# Assignment 2 — Chocolate Bar Recognition: SOTA upgrade
**Continuation of Assignment 1** (4 chocolate bars: Kazakhstan, KitKat, Milky Way, Snickers; 800 frames from 16 self-recorded videos, 4 locations; test = a location the model never saw).

**What A2 adds on top of the A1 pipeline**
| Part | Method | Why |
|---|---|---|
| SOTA backbone | **DINOv2 ViT-S/14 with registers** (Meta, self-supervised; registers: Darcet et al., ICLR 2024) | strong general features → better transfer to unseen locations, native embeddings for nearest-neighbour retrieval |
| SOTA data synthesis | **SAM 2.1** (Meta, 2024) masks → bar cut out and pasted on new texture backgrounds (DTD) | attacks A1's main weakness: generalisation to new surfaces / backgrounds |
| SOTA zero-shot reference | **SigLIP** (Google) text–image model, no training | shows what a foundation model does *without* our dataset |
| Evaluation | leave-one-location-out (4 folds), stress test (blur / dark / partial view / occlusion / noise), Top-2, retrieval P@5, latency | A1 had a single test location and 97% accuracy — robustness is where the upgrade must be shown |
| Explainability | Grad-CAM (CNN + ViT) and nearest-neighbour gallery | required visual explanation |
| Ablation | with vs without SAM 2 synthetic data, for both architectures | required ablation |

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. Full run ≈ 30–45 min (set `RUN_LOLO = False` for a quick ≈ 10 min run on the A1 split only).

## 0. Configuration

In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/my_data"   # same input as A1: folder (or .zip) with one sub-folder per class (videos)
                                                # an A1 `dataset.zip` (extracted frames) also works

FRAMES_PER_CLASS = 200     # identical to A1 -> identical dataset
STORE_SIZE = 256
IMG_SIZE = 224
BATCH = 32
SEED = 42

HEAD_EPOCHS = 10           # stage 1: frozen backbone, train the head  (same recipe as A1)
FT_EPOCHS = 8              # stage 2: unfreeze the top of the backbone
HEAD_LR, FT_LR = 1e-3, 1e-5

RUN_LOLO = True            # leave-one-location-out over all 4 locations (False = only the A1 split)
SYNTH_PER_IMAGE = 1        # synthetic composites per real training image (-> ~50% synthetic share)
N_BACKGROUNDS = 300

DINO_ID = "facebook/dinov2-with-registers-small"   # DINOv2 + 4 register tokens (Darcet et al., ICLR 2024): no artifact tokens -> clean heatmaps
SIGLIP_ID = "google/siglip-base-patch16-224"
SAM2_ID = "facebook/sam2.1-hiera-small"

CLASS_PROMPTS = {          # used only by the SigLIP zero-shot reference
    "kazakhstan": "a photo of a Kazakhstan chocolate bar in a blue wrapper",
    "kitkat": "a photo of a KitKat chocolate bar",
    "milkyway": "a photo of a Milky Way chocolate bar",
    "snickers": "a photo of a Snickers chocolate bar",
}

WORK = "/content/a2"
RAW_DIR, DATA_DIR = f"{WORK}/raw", f"{WORK}/dataset"
FIG_DIR, EXPORT_DIR = f"{WORK}/figures", f"{WORK}/export"
NUM_WORKERS = 2

## 1. Setup

In [ ]:
# SAM 2 (official repo, no CUDA extension needed)
!SAM2_BUILD_CUDA=0 pip -q install "git+https://github.com/facebookresearch/sam2.git"

In [ ]:
import os, glob, json, random, shutil, zipfile, time, copy, platform, math
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance, ImageFont
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from tqdm.auto import tqdm
from transformers import AutoModel

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
for d in (WORK, FIG_DIR, EXPORT_DIR): os.makedirs(d, exist_ok=True)
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
print("torch", torch.__version__, "| device:", DEVICE, "|", GPU_NAME)

### Model loaders
All pretrained models are loaded through these functions (DINOv2, MobileNetV2, SigLIP, SAM 2).

In [ ]:
def load_dino_backbone():
    return AutoModel.from_pretrained(DINO_ID)

def load_mnv2_features():
    return torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").features

_SIGLIP = {}
def siglip_logits(pil_images, prompts):
    """Zero-shot SigLIP: image-text similarity logits, shape (n_images, n_prompts)."""
    from transformers import AutoModel, AutoProcessor
    if not _SIGLIP:
        _SIGLIP["model"] = AutoModel.from_pretrained(SIGLIP_ID).to(DEVICE).eval()
        _SIGLIP["proc"] = AutoProcessor.from_pretrained(SIGLIP_ID)
    model, proc = _SIGLIP["model"], _SIGLIP["proc"]
    out = []
    for i in range(0, len(pil_images), BATCH):
        inp = proc(text=prompts, images=pil_images[i:i + BATCH], padding="max_length", return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out.append(model(**inp).logits_per_image.float().cpu().numpy())
    return np.concatenate(out)

def sam2_candidate_masks(np_images):
    """SAM 2.1, 3x3 grid of single-point prompts x 3 masks = 27 candidate masks + scores per image."""
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    predictor = SAM2ImagePredictor.from_pretrained(SAM2_ID, device=DEVICE)
    out = []
    for im in tqdm(np_images, desc="SAM 2"):
        h, w = im.shape[:2]
        pts = [(fx, fy) for fy in (0.25, 0.5, 0.75) for fx in (0.25, 0.5, 0.75)]
        ms, ss = [], []
        with torch.inference_mode():
            predictor.set_image(im)
            for fx, fy in pts:
                masks, scores, _ = predictor.predict(point_coords=np.array([[w * fx, h * fy]]),
                                                     point_labels=np.array([1]), multimask_output=True)
                ms.append(masks.astype(bool)); ss.append(scores)
        out.append((np.concatenate(ms), np.concatenate(ss)))
    return out

## 2. Data — same frames and the same split as A1
The videos are turned into frames exactly as in A1 (50 windows per video, sharpest frame per window, centre crop, 256×256).
Each video = one location, so `location` = index of the video inside its class (all four bars were filmed at one location before moving on).

In [ ]:
VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv", ".3gp", ".m4v", ".webm"}
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
is_media = lambda f: os.path.splitext(f)[1].lower() in VIDEO_EXT | IMAGE_EXT

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

shutil.rmtree(RAW_DIR, ignore_errors=True)
if os.path.isdir(DRIVE_PATH):
    shutil.copytree(DRIVE_PATH, os.path.join(RAW_DIR, "data"))
else:
    with zipfile.ZipFile(DRIVE_PATH) as z: z.extractall(RAW_DIR)

def find_root(path):
    for root, dirs, _ in os.walk(path):
        dirs[:] = [d for d in dirs if not d.startswith((".", "__MACOSX"))]
        with_media = [d for d in dirs if any(is_media(f) for f in os.listdir(os.path.join(root, d)))]
        if len(with_media) >= 2:
            return root
    raise RuntimeError("No class folders with videos/images found")

SRC_ROOT = find_root(RAW_DIR)
CLASSES = sorted(d for d in os.listdir(SRC_ROOT)
                 if os.path.isdir(os.path.join(SRC_ROOT, d)) and not d.startswith((".", "__MACOSX")))
N_CLS = len(CLASSES)
print("classes:", CLASSES)

In [ ]:
def square(img, size=STORE_SIZE):
    h, w = img.shape[:2]; s = min(h, w)
    y, x = (h - s) // 2, (w - s) // 2
    return cv2.resize(img[y:y + s, x:x + s], (size, size), interpolation=cv2.INTER_AREA)

def sharpness(img):
    return cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

def extract_video(path, n_frames, out_dir, stem):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        print("  !! cannot read", path); return 0
    win = max(1, total // n_frames)
    best, best_s, saved, i = None, -1, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        small = cv2.resize(frame, (320, int(320 * frame.shape[0] / frame.shape[1])))
        s = sharpness(small)
        if s > best_s: best, best_s = frame, s
        i += 1
        if i % win == 0 and saved < n_frames:
            cv2.imwrite(f"{out_dir}/{stem}__f{saved:04d}.jpg", square(best)); saved += 1
            best, best_s = None, -1
    cap.release()
    return saved

shutil.rmtree(DATA_DIR, ignore_errors=True)
for c in CLASSES:
    src, out = os.path.join(SRC_ROOT, c), os.path.join(DATA_DIR, c)
    os.makedirs(out, exist_ok=True)
    files_ = sorted(f for f in os.listdir(src) if is_media(f))
    vids = [f for f in files_ if os.path.splitext(f)[1].lower() in VIDEO_EXT]
    imgs = [f for f in files_ if os.path.splitext(f)[1].lower() in IMAGE_EXT]
    for f in imgs:   # A1 dataset.zip frames keep their "<video>__fNNNN" name; other photos -> group "photos"
        img = cv2.imread(os.path.join(src, f))
        if img is None: continue
        name = f if "__" in f else f"photos__{f}"
        cv2.imwrite(os.path.join(out, os.path.splitext(name)[0] + ".jpg"), square(img))
    per_video = int(np.ceil(max(0, FRAMES_PER_CLASS - len(imgs)) / max(1, len(vids))))
    for f in vids:
        stem = os.path.splitext(f)[0].replace("__", "_")
        extract_video(os.path.join(src, f), per_video, out, stem)

rows = [{"path": os.path.join(DATA_DIR, c, f), "label": c, "group": f.split("__")[0]}
        for c in CLASSES for f in sorted(os.listdir(os.path.join(DATA_DIR, c)))]
df = pd.DataFrame(rows)
df["y"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
loc_map = {(c, g): i for c in CLASSES for i, g in enumerate(sorted(df[df.label == c].group.unique()))}
df["location"] = [loc_map[(c, g)] for c, g in zip(df.label, df.group)]
df["synthetic"] = False
N_LOC = int(df.groupby("label").location.nunique().min())
MAIN_LOC = N_LOC - 1          # A1 test set = last video of every class
assert N_LOC >= 2, "need >= 2 videos (locations) per class"
print(pd.crosstab(df.label, df.location, margins=True))

IMGS = {p: Image.open(p).convert("RGB") for p in df.path}   # 800 small images -> keep in RAM

In [ ]:
def make_fold(test_loc):
    """Same logic as A1: one location = test, the rest split 85/15 into train/val (val only for early stopping)."""
    test = df[df.location == test_loc]
    pool = df[df.location != test_loc]
    train, val = train_test_split(pool, test_size=0.15, stratify=pool.y, random_state=SEED)
    return train, val, test

tr_main, va_main, te_main = make_fold(MAIN_LOC)
split_table = pd.DataFrame({n: d.label.value_counts().reindex(CLASSES)
                            for n, d in [("train", tr_main), ("val", va_main), ("test", te_main)]})
split_table.loc["total"] = split_table.sum()
print("A1 split (test = location", MAIN_LOC, "):"); split_table

## 3. SOTA data synthesis — SAM 2 cut-out + new backgrounds
A1's weakness is the **background/location shift**: the model was trained on 3 surfaces and tested on a 4th.
SAM 2.1 segments the bar: 9 point prompts (3×3 grid) × 3 candidate masks; a candidate is accepted only if it looks like a bar —
one compact, almost rectangular blob (area / min-rotated-rectangle ≥ 0.85), 3–60 % of the image, not covering the border.
This rejects masks of the table/carpet and of the hand. The most rectangular accepted mask wins; frames without one get no synthetic copy.
the bar is pasted at a random scale/position onto a texture from the **Describable Textures Dataset (DTD)**.
Every composite is stored as *synthetic* and is only ever used for **training** (never validation/test).

In [ ]:
def procedural_bg(rng, S=STORE_SIZE):
    a, b = rng.integers(0, 256, 3), rng.integers(0, 256, 3)
    t = np.linspace(0, 1, S)[None, :, None] if rng.random() < .5 else np.linspace(0, 1, S)[:, None, None]
    img = (a * (1 - t) + b * t) * np.ones((S, S, 3))
    img += rng.normal(0, rng.uniform(5, 30), (S, S, 3))
    img = cv2.GaussianBlur(np.clip(img, 0, 255).astype(np.uint8), (0, 0), rng.uniform(0.5, 3))
    return Image.fromarray(img)

def load_backgrounds(n=N_BACKGROUNDS):
    rng = np.random.default_rng(SEED)
    try:
        dtd = torchvision.datasets.DTD(f"{WORK}/dtd", split="train", download=True)
        idx = rng.choice(len(dtd), size=min(n, len(dtd)), replace=False)
        return [dtd[int(i)][0].convert("RGB") for i in idx], "DTD (Describable Textures Dataset)"
    except Exception as e:
        print("DTD download failed -> procedural backgrounds:", repr(e)[:120])
        return [procedural_bg(rng) for _ in range(n)], "procedural gradients/noise"

BGS, BG_SOURCE = load_backgrounds()
print(len(BGS), "backgrounds from", BG_SOURCE)

In [ ]:
def border_frac(m):
    return np.concatenate([m[0], m[-1], m[:, 0], m[:, -1]]).mean()

def shape_stats(m):
    """rectangularity = mask area / area of its minimum rotated rectangle; main = share of the largest component."""
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    if n < 2: return 0.0, 0.0
    k = 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])
    main = st[k, cv2.CC_STAT_AREA] / m.sum()
    cnts, _ = cv2.findContours((lab == k).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    (_, _), (rw, rh), _ = cv2.minAreaRect(max(cnts, key=cv2.contourArea))
    return st[k, cv2.CC_STAT_AREA] / max(rw * rh, 1.0), main

def pick_mask(masks, scores, lo=0.03, hi=0.6, max_border=0.15, min_score=0.6, min_rect=0.85, min_main=0.95):
    """A chocolate bar is one compact, almost rectangular blob that does not cover the image border.
    This rejects background/table masks (large, fragmented, touch the border) and hands (not rectangular).
    Among the valid candidates the most rectangular one wins (ties broken by SAM score)."""
    best, best_key = None, None
    for m, s in zip(masks, scores):
        if not (lo <= m.mean() <= hi and border_frac(m) <= max_border and s >= min_score): continue
        rect, main = shape_stats(m)
        if rect < min_rect or main < min_main: continue
        key = (round(rect, 2), s)
        if best_key is None or key > best_key:
            best, best_key = m, key
    return best

def grabcut_mask(im):
    mask = np.zeros(im.shape[:2], np.uint8)
    S = im.shape[0]; r = int(S * 0.1)
    bgd, fgd = np.zeros((1, 65)), np.zeros((1, 65))
    cv2.grabCut(cv2.cvtColor(im, cv2.COLOR_RGB2BGR), mask, (r, r, S - 2 * r, S - 2 * r), bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
    return pick_mask([np.isin(mask, [1, 3])], [1.0])

paths_all = df.path.tolist()
arrs = [np.array(IMGS[p]) for p in paths_all]
t0 = time.time()
try:
    cands = sam2_candidate_masks(arrs)
    MASKS = {p: pick_mask(m, s) for p, (m, s) in zip(paths_all, cands)}
    MASK_METHOD = f"SAM 2.1 ({SAM2_ID}), 3x3 point prompts + shape filter"
except Exception as e:
    print("!! SAM 2 not available -> GrabCut fallback:", repr(e)[:200])
    MASKS = {p: grabcut_mask(a) for p, a in tqdm(zip(paths_all, arrs), total=len(arrs))}
    MASK_METHOD = "OpenCV GrabCut (fallback)"
MASK_TIME = time.time() - t0
ok = sum(m is not None for m in MASKS.values())
print(f"{MASK_METHOD}: usable mask for {ok}/{len(MASKS)} images ({MASK_TIME:.0f}s)")

In [ ]:
def composite(img, mask, bg, rng, S=STORE_SIZE):
    # background: random crop of a random-scaled texture
    bw, bh = bg.size; k = S * rng.uniform(1.0, 1.6) / min(bw, bh)
    bgr = bg.resize((max(S, int(bw * k)), max(S, int(bh * k))))
    x0, y0 = rng.integers(0, bgr.size[0] - S + 1), rng.integers(0, bgr.size[1] - S + 1)
    canvas = np.array(bgr.crop((x0, y0, x0 + S, y0 + S))).astype(np.float32)
    canvas = np.clip(canvas * rng.uniform(0.7, 1.2), 0, 255)
    # object: crop the mask's bounding box, rescale, paste at a random position with a soft edge
    ys, xs = np.where(mask)
    obj = np.array(img)[ys.min():ys.max() + 1, xs.min():xs.max() + 1].astype(np.float32)
    m = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1].astype(np.float32)
    s = min(rng.uniform(0.7, 1.1), (S - 2) / obj.shape[0], (S - 2) / obj.shape[1])
    nh, nw = max(4, int(obj.shape[0] * s)), max(4, int(obj.shape[1] * s))
    obj, m = cv2.resize(obj, (nw, nh)), cv2.GaussianBlur(cv2.resize(m, (nw, nh)), (5, 5), 0)[..., None]
    py, px = rng.integers(0, S - nh + 1), rng.integers(0, S - nw + 1)
    region = canvas[py:py + nh, px:px + nw]
    canvas[py:py + nh, px:px + nw] = m * obj + (1 - m) * region
    return Image.fromarray(canvas.astype(np.uint8))

def watermark(img, text="SYNTHETIC"):
    img = img.convert("RGB").copy(); d = ImageDraw.Draw(img, "RGBA")
    w, h = img.size; bh = max(14, h // 9)
    d.rectangle([0, h - bh, w, h], fill=(200, 0, 0, 170))
    d.text((6, h - bh + bh // 2 - 6), text, fill=(255, 255, 255, 255))
    return img

rng = np.random.default_rng(SEED)
syn_rows = []
for r in df.itertuples():
    m = MASKS[r.path]
    if m is None: continue
    for k in range(SYNTH_PER_IMAGE):
        key = f"syn::{k}::{r.path}"
        IMGS[key] = composite(IMGS[r.path], m, BGS[rng.integers(len(BGS))], rng)
        syn_rows.append({"path": key, "label": r.label, "group": r.group, "y": r.y,
                         "location": r.location, "synthetic": True, "source": r.path})
syn_df = pd.DataFrame(syn_rows, columns=["path", "label", "group", "y", "location", "synthetic", "source"])
print(len(syn_df), "synthetic images created (used only for training, only from training-fold sources)")

# figure: original | mask | synthetic composite
show = df[df.path.map(lambda p: MASKS[p] is not None)].groupby("label").sample(2, random_state=SEED)
fig, axes = plt.subplots(len(show), 3, figsize=(7.5, 2.4 * len(show)))
for ax_row, r in zip(axes, show.itertuples()):
    im = np.array(IMGS[r.path]); m = MASKS[r.path]
    over = im.copy(); over[m] = (0.5 * over[m] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    syn = IMGS[syn_df[syn_df.source == r.path].path.iloc[0]]
    mask_name = "SAM 2 mask" if MASK_METHOD.startswith("SAM") else "GrabCut mask"
    for ax, x, t in zip(ax_row, [im, over, watermark(syn)], ["original", mask_name, "synthetic composite"]):
        ax.imshow(x); ax.set_title(f"{r.label}: {t}" if t == "original" else t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/synthetic_examples.png", dpi=130); plt.show()

## 4. Models
* **MobileNetV2** (A1 improved model, re-implemented in PyTorch with the same two-stage recipe: frozen head training, then top layers fine-tuned) — the **baseline** for A2.
* **DINOv2 ViT-S/14 with registers** — 4 extra register tokens absorb the high-norm artifact tokens of plain DINOv2, so patch features and heatmaps are clean. Frozen backbone + linear head on `[CLS ; mean(patch tokens)]` (standard DINOv2 linear-probe features), then the last 2 transformer blocks fine-tuned.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([T.RandomHorizontalFlip(), T.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
                      T.ColorJitter(brightness=0.2, contrast=0.2), T.Resize((IMG_SIZE, IMG_SIZE)),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, images, ys, tf): self.images, self.ys, self.tf = images, ys, tf
    def __len__(self): return len(self.images)
    def __getitem__(self, i): return self.tf(self.images[i]), self.ys[i]

def make_loader(rows, tf, shuffle=False):
    ds = ImgDS([IMGS[p] for p in rows.path], rows.y.values.astype("int64"), tf)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")

class MNV2Classifier(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        self.features = load_mnv2_features()
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, n_cls))
    def embed(self, x): return F.adaptive_avg_pool2d(self.features(x), 1).flatten(1)
    def forward(self, x): return self.head(self.embed(x))

class DinoClassifier(nn.Module):
    def __init__(self, n_cls, backbone=None):
        super().__init__()
        self.backbone = load_dino_backbone() if backbone is None else backbone
        d = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * d, n_cls))
        self.n_skip = 1 + getattr(self.backbone.config, "num_register_tokens", 0)   # CLS + register tokens
    def embed(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state          # (B, 1 + registers + patches, d), layer-normed
        return torch.cat([h[:, 0], h[:, self.n_skip:].mean(1)], dim=1)
    def forward(self, x): return self.head(self.embed(x))

def build(arch):
    return (MNV2Classifier if arch == "mnv2" else DinoClassifier)(N_CLS).to(DEVICE)

def set_stage(model, arch, stage):
    for p in model.parameters(): p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = True
    if stage == 2:
        top = list(model.features[-4:]) if arch == "mnv2" else list(model.backbone.encoder.layer[-2:]) + [model.backbone.layernorm]
        for mod in top:
            for p in mod.parameters(): p.requires_grad = True
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def bn_eval(model):   # like Keras base(training=False): BatchNorm statistics stay frozen
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()

@torch.no_grad()
def predict_images(model, images):
    model.eval(); probs, embs = [], []
    loader = DataLoader(ImgDS(images, np.zeros(len(images), "int64"), eval_tf), batch_size=BATCH, num_workers=NUM_WORKERS)
    for x, _ in loader:
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            e = model.embed(x.to(DEVICE)); logits = model.head(e)
        probs.append(F.softmax(logits.float(), 1).cpu().numpy()); embs.append(e.float().cpu().numpy())
    return np.concatenate(probs), np.concatenate(embs)

def predict_rows(model, rows): return predict_images(model, [IMGS[p] for p in rows.path])

def fit(model, train_rows, val_rows, lr, epochs, patience, hist, stage):
    tl, vl = make_loader(train_rows, train_tf, shuffle=True), make_loader(val_rows, eval_tf)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        model.train(); bn_eval(model)
        tot, correct, n = 0.0, 0, 0
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                out = model(x); loss = F.cross_entropy(out, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
        model.eval(); vt, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                    out = model(x)
                vt += F.cross_entropy(out.float(), y, reduction="sum").item(); vc += (out.argmax(1) == y).sum().item(); vn += len(y)
        hist.append({"epoch": len(hist), "stage": stage, "loss": tot / n, "accuracy": correct / n,
                     "val_loss": vt / vn, "val_accuracy": vc / vn})
        if vt / vn < best - 1e-4:
            best, best_state, bad = vt / vn, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)

def train_config(arch, train_rows, val_rows, use_syn):
    seed_all()
    if use_syn:
        train_rows = pd.concat([train_rows, syn_df[syn_df.source.isin(train_rows.path)]])
    model, hist = build(arch), []
    t0 = time.time()
    set_stage(model, arch, 1); fit(model, train_rows, val_rows, HEAD_LR, HEAD_EPOCHS, 4, hist, 1)
    set_stage(model, arch, 2); fit(model, train_rows, val_rows, FT_LR, FT_EPOCHS, 3, hist, 2)
    return model, hist, time.time() - t0, len(train_rows), int(train_rows.synthetic.sum())

## 5. Metrics
Accuracy, macro precision/recall/F1, **Top-2 accuracy** (Top-5 is meaningless with 4 classes),
**retrieval P@5** (share of the 5 nearest *training* images, by cosine similarity of the model embedding, with the same class).

In [ ]:
def compute_metrics(y, probs, emb_q=None, emb_g=None, y_g=None, k=5):
    pred = probs.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    top2 = np.mean([yy in np.argsort(-pp)[:2] for yy, pp in zip(y, probs)])
    out = {"accuracy": accuracy_score(y, pred), "precision": p, "recall": r, "f1": f1, "top2": top2}
    if emb_q is not None:
        q = emb_q / np.linalg.norm(emb_q, axis=1, keepdims=True); g = emb_g / np.linalg.norm(emb_g, axis=1, keepdims=True)
        nn_idx = np.argsort(-(q @ g.T), axis=1)[:, :k]
        out["p_at_5"] = float((y_g[nn_idx] == y[:, None]).mean())
    return out

## 6. Experiments: 2 architectures × with/without SAM 2 synthetic data × 4 test locations
Fold `MAIN_LOC` is exactly the A1 split, so its numbers are directly comparable to A1.

In [ ]:
CONFIGS = [("MobileNetV2 (A1 baseline)", "mnv2", False),
           ("MobileNetV2 + SAM2-synthetic", "mnv2", True),
           ("DINOv2-S", "dino", False),
           ("DINOv2-S + SAM2-synthetic", "dino", True)]
FOLDS = [MAIN_LOC] + [k for k in range(N_LOC) if k != MAIN_LOC] if RUN_LOLO else [MAIN_LOC]

RESULTS, MAIN = [], {}
for k in FOLDS:
    tr, va, te = make_fold(k)
    for name, arch, syn in CONFIGS:
        model, hist, t, n_train, n_syn = train_config(arch, tr, va, syn)
        probs, emb_te = predict_rows(model, te)
        _, emb_tr = predict_rows(model, tr)
        m = compute_metrics(te.y.values, probs, emb_te, emb_tr, tr.y.values)
        RESULTS.append({"model": name, "test_location": k, **m, "train_time_s": t, "n_train": n_train, "n_synthetic": n_syn})
        print(f"loc {k} | {name:30s} acc={m['accuracy']:.3f} F1={m['f1']:.3f} top2={m['top2']:.3f} "
              f"P@5={m['p_at_5']:.3f} | {t:.0f}s, train={n_train} ({n_syn} synthetic)")
        if k == MAIN_LOC:
            MAIN[name] = {"model": model, "arch": arch, "hist": hist, "probs": probs, "emb_te": emb_te, "emb_tr": emb_tr}
        else:
            del model
        if DEVICE == "cuda": torch.cuda.empty_cache()
res = pd.DataFrame(RESULTS)

### 6.1 SigLIP zero-shot reference (no training at all)

In [ ]:
try:
    prompts = [CLASS_PROMPTS.get(c, f"a photo of a {c} chocolate bar") for c in CLASSES]
    logits = siglip_logits([IMGS[p] for p in df.path], prompts)
    zs_probs = np.exp(logits - logits.max(1, keepdims=True)); zs_probs /= zs_probs.sum(1, keepdims=True)
    for k in FOLDS:
        idx = (df.location == k).values
        m = compute_metrics(df.y.values[idx], zs_probs[idx])
        RESULTS.append({"model": "SigLIP zero-shot (no training)", "test_location": k, **m, "train_time_s": 0.0,
                        "n_train": 0, "n_synthetic": 0})
    MAIN["SigLIP zero-shot (no training)"] = {"probs": zs_probs[(df.location == MAIN_LOC).values]}
    print(classification_report(df.y, zs_probs.argmax(1), target_names=CLASSES, digits=3, zero_division=0))
except Exception as e:
    print("SigLIP skipped:", repr(e)[:200])
res = pd.DataFrame(RESULTS)

## 7. Results
### 7.1 A1 split (test = unseen location, identical to A1)

In [ ]:
A1_REF = pd.DataFrame([  # numbers from the A1 report (TensorFlow implementation)
    {"model": "A1: SVM (HOG)", "accuracy": 0.230, "f1": 0.231},
    {"model": "A1: simple CNN", "accuracy": 0.500, "f1": 0.333},
    {"model": "A1: MobileNetV2 (Keras)", "accuracy": 0.970, "f1": 0.970}])
main_tab = res[res.test_location == MAIN_LOC].drop(columns="test_location")
main_table = pd.concat([A1_REF, main_tab], ignore_index=True)
cols = ["model", "accuracy", "precision", "recall", "f1", "top2", "p_at_5", "train_time_s", "n_train", "n_synthetic"]
main_table = main_table[cols].round(3)
display(main_table)

### 7.2 Leave-one-location-out (mean ± std over the 4 locations) — the robust comparison

In [ ]:
if len(FOLDS) > 1:
    lolo = res.groupby("model", sort=False)[["accuracy", "f1", "top2"]].agg(["mean", "std"]).round(3)
    display(lolo)
    per_fold = res.pivot_table(index="model", columns="test_location", values="accuracy", sort=False).round(3)
    display(per_fold)
    ax = per_fold.T.plot.bar(figsize=(9, 4), ylim=(0, 1.05), rot=0)
    ax.set_xlabel("held-out location"); ax.set_ylabel("test accuracy"); ax.set_title("Accuracy per held-out location")
    ax.legend(fontsize=8, loc="lower left"); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/lolo_accuracy.png", dpi=150); plt.show()
else:
    lolo = None

### 7.3 Ablation — with vs without SAM 2 synthetic data

In [ ]:
agg = res.groupby("model", sort=False)[["accuracy", "f1"]].mean()
ablation = pd.DataFrame([
    {"architecture": a, "without synthetic F1": agg.loc[n0, "f1"], "with synthetic F1": agg.loc[n1, "f1"],
     "Δ F1": agg.loc[n1, "f1"] - agg.loc[n0, "f1"], "Δ accuracy": agg.loc[n1, "accuracy"] - agg.loc[n0, "accuracy"]}
    for a, n0, n1 in [("MobileNetV2", CONFIGS[0][0], CONFIGS[1][0]), ("DINOv2-S", CONFIGS[2][0], CONFIGS[3][0])]]).round(3)
print(f"averaged over {len(FOLDS)} held-out location(s)"); display(ablation)

### 7.4 Confusion matrices and training curves (A1 split)

In [ ]:
names = list(MAIN.keys())
fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4))
for ax, n in zip(np.atleast_1d(axes), names):
    cm = confusion_matrix(te_main.y, MAIN[n]["probs"].argmax(1), labels=range(N_CLS))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=CLASSES, yticklabels=CLASSES)
    ax.set_title(n, fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/confusion_matrices.png", dpi=150); plt.show()

trained = [n for n in names if "hist" in MAIN[n]]
fig, axes = plt.subplots(2, len(trained), figsize=(4.2 * len(trained), 6.5), squeeze=False)
for j, n in enumerate(trained):
    h = pd.DataFrame(MAIN[n]["hist"])
    h[["accuracy", "val_accuracy"]].plot(ax=axes[0, j], title=n, fontsize=8)
    h[["loss", "val_loss"]].plot(ax=axes[1, j], fontsize=8)
    ft = h.index[h.stage == 2]
    for a in axes[:, j]:
        if len(ft): a.axvline(ft[0] - 0.5, ls="--", c="grey", lw=1)
        a.set_xlabel("epoch")
plt.suptitle("Training curves (dashed line = start of fine-tuning)"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/training_curves.png", dpi=150); plt.show()

## 8. Stress test — conditions that caused A1's errors
Corruptions applied to the held-out test images (A1 split): blur, low light, partial view (bar half out of frame — A1's main error type), occlusion, sensor noise.

In [ ]:
def corrupt(img, kind, rng):
    S = img.size[0]
    if kind == "clean": return img
    if kind == "blur": return img.filter(ImageFilter.GaussianBlur(4))
    if kind == "dark": return ImageEnhance.Brightness(img).enhance(0.3)
    if kind == "partial view":
        c = int(S * 0.55); x, y = rng.choice([0, S - c]), rng.choice([0, S - c])
        return img.crop((x, y, x + c, y + c)).resize((S, S))
    if kind == "occlusion":
        im = img.copy(); w = int(S * 0.45); x, y = rng.integers(S // 8, S - w - S // 8, 2)
        ImageDraw.Draw(im).rectangle([x, y, x + w, y + w], fill=(128, 128, 128)); return im
    if kind == "noise":
        a = np.array(img).astype(np.float32) + rng.normal(0, 25, (S, S, 3))
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

CONDITIONS = ["clean", "blur", "dark", "partial view", "occlusion", "noise"]
stress = {}
for cond in CONDITIONS:
    rng = np.random.default_rng(SEED)
    imgs = [corrupt(IMGS[p], cond, rng) for p in te_main.path]
    for n in trained:
        stress.setdefault(n, {})[cond] = accuracy_score(te_main.y, predict_images(MAIN[n]["model"], imgs)[0].argmax(1))
stress_df = pd.DataFrame(stress).T[CONDITIONS].round(3)
stress_df["mean (corrupted)"] = stress_df[CONDITIONS[1:]].mean(1).round(3)
display(stress_df)

rng = np.random.default_rng(SEED); ex = IMGS[te_main.path.iloc[0]]
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(2.2 * len(CONDITIONS), 2.4))
for ax, c in zip(axes, CONDITIONS): ax.imshow(corrupt(ex, c, rng)); ax.set_title(c, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/stress_conditions.png", dpi=130); plt.show()
ax = stress_df[CONDITIONS].T.plot.bar(figsize=(9, 4), ylim=(0, 1.05), rot=0)
ax.set_ylabel("accuracy"); ax.set_title("Stress test (held-out location)"); ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/stress_test.png", dpi=150); plt.show()

## 9. Explainability — Grad-CAM and nearest neighbours

In [ ]:
def grad_cam(model, arch, img):
    """Grad-CAM for the predicted class. CNN: last conv block. ViT: tokens after norm1 of the last block (patch grid)."""
    model.eval()
    layer = model.features[-1] if arch == "mnv2" else model.backbone.encoder.layer[-1].norm1
    store = {}
    def hook(_, __, out):
        out.retain_grad(); store["a"] = out
    h = layer.register_forward_hook(hook)
    x = eval_tf(img)[None].to(DEVICE).requires_grad_(True)
    logits = model(x); cls = int(logits.argmax(1))
    model.zero_grad(); logits[0, cls].backward(); h.remove()
    a, g = store["a"][0].detach().float(), store["a"].grad[0].float()
    if arch == "mnv2":                                   # (C, H, W)
        cam = F.relu((g.mean((1, 2))[:, None, None] * a).sum(0))
    else:                                                # (CLS + registers + N, C) -> keep only patch tokens
        a, g = a[model.n_skip:], g[model.n_skip:]; s = int(math.sqrt(a.shape[0]))
        cam = F.relu(a @ g.mean(0)).reshape(s, s)
    cam = cam.cpu().numpy(); cam = cam / (cam.max() + 1e-8)
    probs = F.softmax(logits.detach().float(), 1)[0].cpu().numpy()
    return cv2.resize(cam, img.size), cls, probs

def overlay(img, cam, alpha=0.45):
    heat = (plt.cm.jet(cam)[..., :3] * 255).astype(np.float32)
    return Image.fromarray((np.array(img) * (1 - alpha) + heat * alpha).astype(np.uint8))

dino_names = [n for n in trained if MAIN[n]["arch"] == "dino"]
mnv2_names = [n for n in trained if MAIN[n]["arch"] == "mnv2"]
score = lambda n: (lolo.loc[n, ("f1", "mean")] if lolo is not None else main_tab.set_index("model").loc[n, "f1"])
BEST_DINO = max(dino_names, key=score); BEST_MNV2 = max(mnv2_names, key=score)
print("best DINOv2 config:", BEST_DINO, "| best MobileNetV2 config:", BEST_MNV2)

pred_d = MAIN[BEST_DINO]["probs"].argmax(1)
wrong = np.where(pred_d != te_main.y.values)[0].tolist()
right = te_main.reset_index(drop=True).groupby("y").sample(1, random_state=SEED).index.tolist()
pick = (wrong[:2] + right)[:6]
fig, axes = plt.subplots(len(pick), 3, figsize=(7.5, 2.5 * len(pick)), squeeze=False)
for row, i in zip(axes, pick):
    img = IMGS[te_main.path.iloc[i]]
    row[0].imshow(img); row[0].set_title(f"true: {te_main.label.iloc[i]}", fontsize=9)
    for ax, n in zip(row[1:], [BEST_MNV2, BEST_DINO]):
        cam, cls, pr = grad_cam(MAIN[n]["model"], MAIN[n]["arch"], img)
        ax.imshow(overlay(img, cam)); ax.set_title(f"{n.split(' ')[0]}: {CLASSES[cls]} ({pr[cls]:.2f})", fontsize=9,
                                                   color="green" if cls == te_main.y.iloc[i] else "crimson")
    for ax in row: ax.axis("off")
plt.suptitle("Grad-CAM on held-out test images"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/gradcam.png", dpi=130); plt.show()

In [ ]:
eq, eg = MAIN[BEST_DINO]["emb_te"], MAIN[BEST_DINO]["emb_tr"]
eqn, egn = eq / np.linalg.norm(eq, axis=1, keepdims=True), eg / np.linalg.norm(eg, axis=1, keepdims=True)
sims = eqn @ egn.T
q_idx = te_main.reset_index(drop=True).groupby("y").sample(1, random_state=1).index.tolist()
fig, axes = plt.subplots(len(q_idx), 6, figsize=(12, 2.3 * len(q_idx)), squeeze=False)
for row, qi in zip(axes, q_idx):
    row[0].imshow(IMGS[te_main.path.iloc[qi]]); row[0].set_title(f"query: {te_main.label.iloc[qi]}", fontsize=9)
    for ax, gi in zip(row[1:], np.argsort(-sims[qi])[:5]):
        ok_ = tr_main.y.iloc[gi] == te_main.y.iloc[qi]
        ax.imshow(IMGS[tr_main.path.iloc[gi]]); ax.set_title(f"{tr_main.label.iloc[gi]} · {sims[qi, gi]:.2f}", fontsize=8,
                                                             color="green" if ok_ else "crimson")
    for ax in row: ax.axis("off")
plt.suptitle(f"Nearest training images ({BEST_DINO} embedding, cosine similarity)"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/nearest_neighbours.png", dpi=120); plt.show()

if wrong:
    fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(2.2 * min(8, len(wrong)), 2.6), squeeze=False)
    for ax, i in zip(axes[0], wrong[:8]):
        ax.imshow(IMGS[te_main.path.iloc[i]]); ax.axis("off")
        ax.set_title(f"true: {te_main.label.iloc[i]}\npred: {CLASSES[pred_d[i]]}", fontsize=8, color="crimson")
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/misclassified_dino.png", dpi=120); plt.show()
print(f"{BEST_DINO}: {len(wrong)} / {len(te_main)} test images misclassified")

## 10. Latency (batch = 1, average of 30 runs)

In [ ]:
def latency_ms(model, device, n=30):
    m = copy.deepcopy(model).to(device).eval(); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(5): m(x)
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n): m(x)
        if device == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000

lat = []
for n in [BEST_MNV2, BEST_DINO]:
    m = MAIN[n]["model"]
    row = {"model": n, "params (M)": sum(p.numel() for p in m.parameters()) / 1e6, "CPU ms": latency_ms(m, "cpu")}
    if DEVICE == "cuda": row["GPU ms"] = latency_ms(m, "cuda")
    lat.append(row)
lat_df = pd.DataFrame(lat)
lat_df["CPU FPS"] = 1000 / lat_df["CPU ms"]
if "GPU ms" in lat_df: lat_df["GPU FPS"] = 1000 / lat_df["GPU ms"]
ENV = {"gpu": GPU_NAME, "cpu": platform.processor() or platform.machine(), "cpu_threads": torch.get_num_threads(),
       "torch": torch.__version__, "python": platform.python_version()}
print(ENV); display(lat_df.round(2))

## 11. Save results, figures and the demo-app model

In [ ]:
best = MAIN[BEST_DINO]["model"]
os.makedirs(f"{EXPORT_DIR}/model", exist_ok=True)
torch.save({k: v.half() if v.is_floating_point() else v for k, v in best.state_dict().items()},
           f"{EXPORT_DIR}/model/dino_classifier.pt")
best.backbone.config.save_pretrained(f"{EXPORT_DIR}/model/dino_config")

top1 = sims.max(1)
thumbs = np.stack([np.array(IMGS[p].resize((112, 112))) for p in tr_main.path])
np.savez_compressed(f"{EXPORT_DIR}/model/gallery.npz", emb=eg.astype(np.float16), y=tr_main.y.values, thumbs=thumbs)

for sub in ("examples/test", "examples/synthetic"): os.makedirs(f"{EXPORT_DIR}/{sub}", exist_ok=True)
for r in te_main.groupby("label").sample(2, random_state=SEED).itertuples():
    IMGS[r.path].save(f"{EXPORT_DIR}/examples/test/{r.label}_{os.path.basename(r.path)}")
for r in syn_df.groupby("label").sample(2, random_state=SEED).itertuples():
    watermark(IMGS[r.path]).save(f"{EXPORT_DIR}/examples/synthetic/SYNTHETIC_{r.label}_{r.Index}.jpg")

app_cfg = {"classes": CLASSES, "img_size": IMG_SIZE, "mean": MEAN, "std": STD, "model_name": BEST_DINO,
           "nn_threshold": float(np.percentile(top1, 2)), "mask_method": MASK_METHOD, "backgrounds": BG_SOURCE,
           "test_location_metrics": main_tab.set_index("model").loc[BEST_DINO, ["accuracy", "f1", "top2", "p_at_5"]].round(3).to_dict(),
           "lolo": None if lolo is None else {n: {"acc_mean": float(lolo.loc[n, ("accuracy", "mean")]),
                                                  "f1_mean": float(lolo.loc[n, ("f1", "mean")])} for n in lolo.index}}
json.dump(app_cfg, open(f"{EXPORT_DIR}/model/config.json", "w"), indent=2, default=float)

summary = {"classes": CLASSES, "n_locations": N_LOC, "folds": FOLDS, "mask_method": MASK_METHOD,
           "masks_ok": ok, "mask_time_s": MASK_TIME, "backgrounds": BG_SOURCE, "n_synthetic_total": len(syn_df),
           "env": ENV, "results": RESULTS, "stress": stress, "latency": lat, "best_dino": BEST_DINO, "best_mnv2": BEST_MNV2}
json.dump(summary, open(f"{FIG_DIR}/results.json", "w"), indent=2, default=float)
for name_, tab in [("main_table", main_table), ("ablation", ablation), ("stress", stress_df), ("latency", lat_df)]:
    tab.to_csv(f"{FIG_DIR}/{name_}.csv")
if lolo is not None: lolo.to_csv(f"{FIG_DIR}/lolo.csv")

shutil.make_archive(f"{WORK}/figures", "zip", FIG_DIR)
shutil.make_archive(f"{WORK}/app_model", "zip", EXPORT_DIR)
print("saved:", f"{WORK}/figures.zip", f"{WORK}/app_model.zip")
try:
    from google.colab import files
    files.download(f"{WORK}/figures.zip"); files.download(f"{WORK}/app_model.zip")
except ImportError:
    pass